# 🤖 Chatbot using HuggingFace Transformers
**Innomatics Research Labs — NLP Assignment 3**  
**Task:** Build a conversational chatbot using a pre-trained transformer model  
**Model:** DialoGPT-medium (`microsoft/DialoGPT-medium`)  
**Tools:** Python · HuggingFace Transformers · PyTorch

---

## Pipeline
```
User Input → Model Processing → Response Generation → Display Output → Loop Until Exit
```

## Step 1 — Install Dependencies

In [1]:
!pip install transformers torch -q

## Step 2 — Imports

In [2]:
import torch
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cpu


## Step 3 — Model Loading

**Model chosen: `microsoft/DialoGPT-medium`**  
DialoGPT is a large-scale pre-trained dialogue response generation model trained on 147M conversation-like exchanges from Reddit. It is specifically designed for multi-turn conversational tasks — unlike GPT-2 which is a general text generator, DialoGPT is optimized for producing coherent and contextually relevant chat responses.

In [3]:
MODEL_NAME = 'microsoft/DialoGPT-medium'

print('Loading tokenizer and model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

# DialoGPT uses EOS token as padding token
tokenizer.pad_token = tokenizer.eos_token

total_params = sum(p.numel() for p in model.parameters())
print('Model loaded:', MODEL_NAME)
print('Parameters: ', format(total_params, ','))
print('Device:     ', device)

Loading tokenizer and model...


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded: microsoft/DialoGPT-medium
Parameters:  406,286,336
Device:      cpu


## Step 4 — Response Generation Function

**How DialoGPT generates responses:**  
The model takes the full conversation history encoded as a single token sequence. Each turn is separated by the EOS token (`<|endoftext|>`). The model then generates the next turn (the bot's reply) by predicting tokens one by one until it generates an EOS token or hits max length.

**Generation parameters explained:**
- `max_new_tokens` — limits response length
- `temperature` — controls randomness (lower = more focused, higher = more creative)
- `top_p` — nucleus sampling: only sample from tokens making up top 90% probability mass
- `do_sample=True` — enables stochastic sampling (vs greedy decoding)
- `repetition_penalty` — discourages the model from repeating the same phrases

In [4]:
def generate_response(user_input, chat_history_ids=None, max_history_turns=5):
    """
    Generates a chatbot response given user input and conversation history.

    Args:
        user_input:        Raw string from the user
        chat_history_ids:  Tensor of previous conversation tokens (None for first turn)
        max_history_turns: Max past turns to keep in context (prevents token overflow)

    Returns:
        response_text:     The bot's reply as a string
        new_history_ids:   Updated conversation history tensor
    """
    # Encode user input and append EOS token to signal end of user turn
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    ).to(device)

    # Concatenate new input with conversation history
    if chat_history_ids is not None:
        # Trim history to avoid exceeding model's max token limit (1024)
        # Keep only the last max_history_turns * ~50 tokens as rough estimate
        max_history_tokens = 900
        if chat_history_ids.shape[-1] > max_history_tokens:
            chat_history_ids = chat_history_ids[:, -max_history_tokens:]
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_new_tokens=150,
        temperature=0.75,         # Moderate randomness
        top_p=0.9,                # Nucleus sampling
        do_sample=True,           # Stochastic generation
        repetition_penalty=1.3,   # Reduce repetition
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode only the new tokens (bot's reply)
    response_text = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    ).strip()

    return response_text, chat_history_ids


print('Response generation function ready.')

Response generation function ready.


## Step 5 — Simulated Conversation Demo

> **Note:** `input()` is not supported in Jupyter/Colab cells.  
> This cell simulates a full conversation by running a predefined list of user messages  
> through the chatbot and printing the responses — demonstrating all required features:  
> model loading, response generation, conversation history, and exit condition.
> The interactive terminal version is provided in Step 6.

In [5]:
# Predefined conversation to demonstrate the chatbot
demo_inputs = [
    'Hello',
    'What is Artificial Intelligence?',
    'Who created Python?',
    'What is machine learning?',
    'Can you tell me something interesting?',
    'Thank you',
    'exit'
]

print('=' * 55)
print('  DialoGPT Chatbot — Conversation Demo')
print('=' * 55)
print('Chatbot: Hello! I am your AI assistant. How can I help you today?')
print()

chat_history_ids = None  # Start with empty history

for user_input in demo_inputs:
    print('You:', user_input)

    # Check exit condition
    if user_input.lower() in ['exit', 'quit']:
        print('Chatbot: Goodbye! It was great talking to you. Take care!')
        print()
        print('=' * 55)
        print('  Conversation ended.')
        print('=' * 55)
        break

    # Generate response
    response, chat_history_ids = generate_response(user_input, chat_history_ids)
    print('Chatbot:', response)
    print()

  DialoGPT Chatbot — Conversation Demo
Chatbot: Hello! I am your AI assistant. How can I help you today?

You: Hello


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot: Hey there! How are you today? I'm not feeling well, but what about yourself do you think is the cause of your illness?

You: What is Artificial Intelligence?
Chatbot: I have no idea...

You: Who created Python?
Chatbot: ... or Java?

You: What is machine learning?
Chatbot: Why did you bring that up in a thread where nobody would ask?

You: Can you tell me something interesting?
Chatbot: How did you know my name was Alex? I was trying to tell everyone.

You: Thank you
Chatbot: You're welcome, happy holidays.

You: exit
Chatbot: Goodbye! It was great talking to you. Take care!

  Conversation ended.


## Step 6 — Interactive Chatbot (Terminal / Script Version)

The cell below contains the full interactive loop using `input()`.  
This works when run as a `.py` script or in a local Jupyter environment.  
In Colab, use the demo above to see the conversation output.

In [6]:
def run_chatbot():
    """
    Interactive chatbot loop.
    Accepts user input from console, generates responses using DialoGPT,
    maintains conversation history, and exits on 'exit' or 'quit'.
    """
    print('=' * 55)
    print('  DialoGPT Chatbot — Interactive Mode')
    print('  Type "exit" or "quit" to end the conversation.')
    print('=' * 55)
    print('Chatbot: Hello! I am your AI assistant. How can I help you today?')
    print()

    chat_history_ids = None  # Conversation history starts empty

    while True:
        # Accept user input
        user_input = input('You: ').strip()

        # Skip empty inputs
        if not user_input:
            continue

        # Exit condition
        if user_input.lower() in ['exit', 'quit']:
            print('Chatbot: Goodbye! It was great talking to you. Take care!')
            print()
            print('=' * 55)
            print('  Conversation ended.')
            print('=' * 55)
            break

        # Generate and display response
        response, chat_history_ids = generate_response(user_input, chat_history_ids)
        print('Chatbot:', response)
        print()

# Uncomment below to run interactively (works in local Jupyter / terminal)
# run_chatbot()

print('Interactive chatbot function defined. Uncomment run_chatbot() to use.')

Interactive chatbot function defined. Uncomment run_chatbot() to use.


## Step 7 — Analysis & Insights

### How DialoGPT works
DialoGPT is built on the GPT-2 architecture — a transformer decoder that generates text auto-regressively (one token at a time, left to right). Unlike encoder-decoder models (like T5 or BART), it has no separate encoder — the entire conversation history is fed as a single flat sequence and the model predicts the next token based on all preceding context.

### Conversation History Management
Multi-turn memory is handled by concatenating all previous turns into one growing token sequence. Each turn is separated by the EOS token. The model sees the full history and generates a reply conditioned on everything said so far. This is why the history must be trimmed — the model has a hard limit of 1024 tokens.

### Why DialoGPT over GPT-2 for chat
GPT-2 was trained to continue text, not to have conversations. If you ask it a question, it will often continue the question itself rather than answer it. DialoGPT was fine-tuned on Reddit conversations, so it learned to produce turn-based replies rather than text continuations.

### Generation Parameters Tradeoff
| Parameter | Low value | High value |
|---|---|---|
| `temperature` | Focused, repetitive | Creative, unpredictable |
| `top_p` | Safer vocabulary | More diverse words |
| `repetition_penalty` | May loop | More varied output |

### Limitations
- DialoGPT-medium has no factual memory — it cannot reliably answer knowledge questions like "Who created Python?" because it was trained on Reddit, not a knowledge base
- Responses can be inconsistent across turns without fine-tuning on a specific domain
- For factual Q&A, a model like `facebook/blenderbot-400M-distill` or a RAG system would be more appropriate

---
## Summary

| Component | Detail |
|---|---|
| **Model** | microsoft/DialoGPT-medium |
| **Architecture** | GPT-2 decoder, fine-tuned on Reddit conversations |
| **Memory** | Full conversation history concatenated as token sequence |
| **Generation** | Nucleus sampling (top_p=0.9), temperature=0.75 |
| **Exit** | Detects 'exit' or 'quit' keywords |

---
*Completed as part of Innomatics Research Labs — Data Science Internship, February 2026*  
*Tools: Python · HuggingFace Transformers · PyTorch*